In [1]:
import sys
sys.path.append('..')  
from modules.m1_tokenizer import Tokenizer

token = Tokenizer()

c:\Users\harsh\Desktop\PRO\nanoJepa\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer: bigscience/bloom-560m


  type         : TokenizersBackend
  vocab_size   : 250,680
  max_length   : 256
  pad_token    : '<pad>' (id=3)
  padding_side : right

  Verification:
    ✓ [spaces] 'hello world' → 'hello world'
    ✓ [math] '2 + 2 = 4' → '2 + 2 = 4'
    ✓ [symbols] 'x² + √16 = ∑' → 'x² + √16 = ∑'
  All checks passed ✓



In [2]:
# ═══ Cell 2: TEST 1 - Basic Round-Trip ═══
# Can we encode text → numbers → text and get back the same thing?

test_sentences = [
    "2 + 2 = 4",
    "If x² + 5x + 6 = 0, find x",
    "What is the derivative of sin(x)?",
    "Calculate the area of a circle with radius 5",
    "Solve: 3x - 7 = 14"
]

print("TEST 1: Round-Trip Encoding/Decoding")
print("="*50)

all_pass = True
for sentence in test_sentences:
    encoded = token.encode(sentence)
    decoded = token.decode(encoded['input_ids'])
    match = sentence.strip() == decoded.strip()
    status = "✅" if match else "❌"
    print(f"{status} Original: '{sentence}'")
    print(f"   Decoded:  '{decoded}'")
    if not match:
        all_pass = False
    print()

print(f"{'ALL TESTS PASSED ✅' if all_pass else 'SOME TESTS FAILED ❌'}")

# EXPECTED OUTPUT:
# TEST 1: Round-Trip Encoding/Decoding
# ==================================================
# ✅ Original: '2 + 2 = 4'
#    Decoded:  '2 + 2 = 4'
# ...
# ALL TESTS PASSED ✅

TEST 1: Round-Trip Encoding/Decoding
✅ Original: '2 + 2 = 4'
   Decoded:  '2 + 2 = 4'

✅ Original: 'If x² + 5x + 6 = 0, find x'
   Decoded:  'If x² + 5x + 6 = 0, find x'

✅ Original: 'What is the derivative of sin(x)?'
   Decoded:  'What is the derivative of sin(x)?'

✅ Original: 'Calculate the area of a circle with radius 5'
   Decoded:  'Calculate the area of a circle with radius 5'

✅ Original: 'Solve: 3x - 7 = 14'
   Decoded:  'Solve: 3x - 7 = 14'

ALL TESTS PASSED ✅


In [3]:
# ═══ Cell 3: TEST 2 - Math Symbols ═══
# Can the tokenizer handle special math symbols?

math_texts = [
    "∑(i=1 to n)",     # Summation
    "∫ f(x) dx",       # Integral
    "√(x² + y²)",      # Square root
    "x ≤ 5 and x ≥ 0", # Less/greater than or equal
    "π ≈ 3.14159"       # Pi and approximation
]

print("TEST 2: Math Symbol Handling")
print("="*50)

for text in math_texts:
    try:
        encoded = token.encode(text)
        decoded = token.decode(encoded['input_ids'])
        n_tokens = encoded['attention_mask'].sum().item()
        print(f"✅ '{text}' → {int(n_tokens)} tokens → '{decoded}'")
    except Exception as e:
        print(f"❌ '{text}' → ERROR: {e}")

# EXPECTED OUTPUT:
# TEST 2: Math Symbol Handling
# ==================================================
# ✅ '∑(i=1 to n)' → 8 tokens → '∑(i=1 to n)'
# ✅ '∫ f(x) dx' → 7 tokens → '∫ f(x) dx'
# ... etc

TEST 2: Math Symbol Handling
✅ '∑(i=1 to n)' → 8 tokens → '∑(i=1 to n)'
✅ '∫ f(x) dx' → 6 tokens → '∫ f(x) dx'
✅ '√(x² + y²)' → 8 tokens → '√(x² + y²)'
✅ 'x ≤ 5 and x ≥ 0' → 7 tokens → 'x ≤ 5 and x ≥ 0'
✅ 'π ≈ 3.14159' → 6 tokens → 'π ≈ 3.14159'


In [4]:
# ═══ Cell 4: TEST 3 - Batch Encoding ═══
# Can we encode 16 sentences at once, all same length?

batch_sentences = [
    "2 + 2 = ?",
    "What is 5 times 3?",
    "Solve for x: 2x = 10",
    "Find the sum of 1 to 100",
    "What is 15% of 200?",
    "Calculate 7! (factorial)",
    "Is 17 a prime number?",
    "What is the GCD of 12 and 18?",
    "Convert 3/4 to decimal",
    "What is log base 2 of 8?",
    "Simplify: 2x + 3x",
    "What is the slope of y = 3x + 1?",
    "Find the mean of [2, 4, 6, 8]",
    "What is 2 to the power of 10?",
    "Solve: x/3 = 5",
    "What is the perimeter of a square with side 4?"
]

result = token.batch_encode(batch_sentences)

print("TEST 3: Batch Encoding")
print("="*50)
print(f"Number of sentences: {len(batch_sentences)}")
print(f"Input IDs shape:     {result['input_ids'].shape}")
print(f"Attention mask shape: {result['attention_mask'].shape}")
print(f"Expected shape:      torch.Size([16, 256])")
print()

shapes_correct = (
    result['input_ids'].shape == (16, 256) and
    result['attention_mask'].shape == (16, 256)
)
print(f"{'PASS ✅' if shapes_correct else 'FAIL ❌'}: All 16 sentences padded to length 256")

# Show token counts for each sentence:
for i, sent in enumerate(batch_sentences[:5]):  # Show first 5
    n_real = result['attention_mask'][i].sum().item()
    print(f"  Sentence {i+1}: {int(n_real)} real tokens + {256-int(n_real)} padding = 256 total")

# EXPECTED OUTPUT:
# TEST 3: Batch Encoding
# ==================================================
# Number of sentences: 16
# Input IDs shape:     torch.Size([16, 256])
# Attention mask shape: torch.Size([16, 256])
# Expected shape:      torch.Size([16, 256])
#
# PASS ✅: All 16 sentences padded to length 256
#   Sentence 1: 7 real tokens + 249 padding = 256 total
#   Sentence 2: 8 real tokens + 248 padding = 256 total
#   ... etc

TEST 3: Batch Encoding
Number of sentences: 16
Input IDs shape:     torch.Size([16, 256])
Attention mask shape: torch.Size([16, 256])
Expected shape:      torch.Size([16, 256])

PASS ✅: All 16 sentences padded to length 256
  Sentence 1: 5 real tokens + 251 padding = 256 total
  Sentence 2: 6 real tokens + 250 padding = 256 total
  Sentence 3: 7 real tokens + 249 padding = 256 total
  Sentence 4: 7 real tokens + 249 padding = 256 total
  Sentence 5: 6 real tokens + 250 padding = 256 total


In [5]:
# ═══ Cell 5: TEST 4 - Truncation ═══
# Does a very long text get cut to 256 tokens?

# Create a very long text (definitely more than 256 tokens)
long_text = "The sum of all natural numbers from 1 to n is given by the formula n times n plus one divided by two. " * 20

encoded = token.encode(long_text)

print("TEST 4: Truncation")
print("="*50)
print(f"Original text length: {len(long_text)} characters")
print(f"Token IDs length:     {len(encoded['input_ids'])} (should be 256)")
print(f"All mask values = 1:  {encoded['attention_mask'].sum().item() == 256}")
print(f"{'PASS ✅' if len(encoded['input_ids']) == 256 else 'FAIL ❌'}: Truncated to 256")

# EXPECTED OUTPUT:
# TEST 4: Truncation
# ==================================================
# Original text length: 2000 characters
# Token IDs length:     256 (should be 256)
# All mask values = 1:  True
# PASS ✅: Truncated to 256

TEST 4: Truncation
Original text length: 2040 characters
Token IDs length:     256 (should be 256)
All mask values = 1:  True
PASS ✅: Truncated to 256


In [6]:
# ═══ Cell 6: TEST 5 - Edge Cases ═══

print("TEST 5: Edge Cases")
print("="*50)

# Empty string
try:
    result = token.encode("")
    print(f"✅ Empty string handled. Shape: {result['input_ids'].shape}")
except Exception as e:
    print(f"❌ Empty string error: {e}")

# Single character
try:
    result = token.encode("5")
    decoded = token.decode(result['input_ids'])
    print(f"✅ Single char '5' → decoded '{decoded.strip()}'")
except Exception as e:
    print(f"❌ Single char error: {e}")

# Only numbers
try:
    result = token.encode("1234567890")
    decoded = token.decode(result['input_ids'])
    print(f"✅ Numbers '1234567890' → decoded '{decoded.strip()}'")
except Exception as e:
    print(f"❌ Numbers error: {e}")

print()
print("="*50)
print("ALL TOKENIZER TESTS COMPLETE!")
print("="*50)

TEST 5: Edge Cases
❌ Empty string error: encode() got empty string.
✅ Single char '5' → decoded '5'
✅ Numbers '1234567890' → decoded '1234567890'

ALL TOKENIZER TESTS COMPLETE!
